# BNPL Default Risk: Interactive Inference

## Workflow Overview
This notebook shows how to use the trained PyTorch model to make real time predictions on single customer records. The workflow is:

1. Load the saved model and transformers from `models/best_pytorch_model.pth` and `models/transformers/`
2. Define the numeric and categorical feature columns, with their ranges and options
3. Take user input for each feature, with validation on the ranges and categories
4. Preprocess the input by scaling numerics and encoding categoricals with the saved transformers, which were fit on the training data
5. Run the model and display the predicted risk class along with the class probabilities
6. Allow the user to make another prediction or exit

## Why Interactive
This mimics a simple deployment scenario where a user with a customer's details wants an instant risk assessment. It also checks that the saved model and preprocessing pipeline still work outside the training environment.

### Imports and Configuration
Setting up the notebook environment and importing the inference helpers from `src/inference.py`.

In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import torch

import utils as U
from config import CAT_ENCODER_PATH, MODEL_CONFIG_PATH, MODEL_PATH, SCALER_PATH
from inference import display_prediction, get_user_input, process_single_input

### Load Saved Components
Loading the trained model, scaler, and category mappings, then moving the model to the available device.

In [6]:
model, scaler, cat_mappings = U.load_inference_components(
    MODEL_PATH, SCALER_PATH, CAT_ENCODER_PATH, MODEL_CONFIG_PATH
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

EmbeddingNet(
  (embeddings): ModuleList(
    (0): Embedding(4, 8)
    (1): Embedding(2, 8)
    (2): Embedding(5, 8)
  )
  (net): Sequential(
    (0): Linear(in_features=31, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.4, inplace=False)
    (8): Linear(in_features=64, out_features=3, bias=True)
  )
)

### Interactive Prediction
Prompts for one customer's features and returns the predicted risk class with per-class probabilities. Input validation and range checks live in `get_user_input` inside `inference.py`.

In [7]:
user_data = get_user_input(cat_mappings)
pred_class, probs = process_single_input(
    user_data, model, scaler, cat_mappings, device
)
display_prediction(user_data, pred_class, probs)

Enter customer details for default risk prediction.

Employment_Status options: ['Employed', 'Freelancer', 'Student', 'Unemployed']

Late_Payment_History options: ['No', 'Yes']

Shopping_Category_Most_Frequent options: ['Electronics', 'Fashion', 'Groceries/Essentials', 'Home/Furniture', 'Travel']

Inputs:
  Age: 32.0
  Total_BNPL_Active_Loans: 2.0
  Total_BNPL_Debt_USD: 23500.0
  Average_Transaction_Value_USD: 125.0
  Income_USD: 145000.0
  Credit_Score: 815.0
  Employment_Status: Unemployed
  Late_Payment_History: Yes
  Shopping_Category_Most_Frequent: Travel

Predicted Default Risk: High (confidence: 91.71%)
Class probabilities:
  Low: 0.00%
  Medium: 8.29%
  High: 91.71%


In [8]:
while True:
    again = input("\nMake another prediction? (y/n): ").strip().lower()
    if again not in ("y", "yes"):
        print("Exiting the prediction tool.")
        break
    user_data = get_user_input(cat_mappings)
    pred_class, probs = process_single_input(
        user_data, model, scaler, cat_mappings, device
    )
    display_prediction(user_data, pred_class, probs)

Exiting the prediction tool.
